### **Road Data Processing**

In [1]:
import qgis

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString

In [3]:
file_path = "../datasets/raw/taita_taveta_roads.shp"
gdf = gpd.read_file(file_path)
gdf.head()

,name,name_en,highway,surface,smoothness,width,lanes,oneway,bridge,layer,source,name_sw,osm_id,osm_type,geometry
0,None,None,path,None,None,None,None,None,None,None,None,None,909925510,ways_line,"LINESTRING (38.36846 -3.41544, 38.36846 -3.415..."
1,None,None,track,None,None,None,None,None,None,None,None,None,502298631,ways_line,"LINESTRING (38.30726 -3.56813, 38.30713 -3.567..."
2,None,None,track,None,None,None,None,None,None,None,None,None,502298704,ways_line,"LINESTRING (38.38066 -3.66766, 38.37747 -3.665..."
3,None,None,track,None,None,None,None,None,None,None,None,None,502298782,ways_line,"LINESTRING (38.35024 -3.66972, 38.35131 -3.671..."
4,None,None,track,None,None,None,None,None,None,None,None,None,502064660,ways_line,"LINESTRING (38.48848 -3.53727, 38.48877 -3.537..."


In [4]:
# Ensure highway column is not null, as it is the basis for imputation
# Dropping rows where 'highway' is null is generally safe for road networks
gdf.dropna(subset=['highway'], inplace=True)

gdf = gdf.rename(columns={'highway': 'road_class'})
gdf['maxspeed'] = None
gdf = gdf[['osm_id', 'road_class', 'surface', 'maxspeed', 'geometry']]

print(f"Total road segments: {len(gdf)}")

Total road segments: 9985


In [5]:
print("\nMissing values before imputation:")
print(gdf.isnull().sum())


Missing values before imputation:
osm_id           0
road_class       0
surface       7899
maxspeed      9985
geometry         0
dtype: int64


In [6]:
# Reproject for accurate length calculation
utm_crs = 'EPSG:32737'
gdf_proj = gdf.to_crs(utm_crs)

# Calculate length in meters (since the CRS unit is meters)
gdf_proj['length_m'] = gdf_proj.geometry.length

# Convert to kilometers
gdf_proj['length_km'] = gdf_proj['length_m'] / 1000

# Drop the intermediate meter column
gdf = gdf_proj.drop(columns=['length_m'])

In [7]:
gdf.head()

,osm_id,road_class,surface,maxspeed,geometry,length_km
0,909925510,path,None,None,"LINESTRING (429848.389 9622463.783, 429848.384...",0.754109
1,502298631,track,None,None,"LINESTRING (423061.53 9605580.562, 423047.982 ...",1.144575
2,502298704,track,None,None,"LINESTRING (431222.357 9594584.113, 430867.313...",1.436449
3,502298782,track,None,None,"LINESTRING (427843.773 9594354.07, 427962.456 ...",1.868584
4,502064660,track,None,None,"LINESTRING (443187.058 9609004.785, 443220.074...",0.988785


In [8]:
# Impute Missing surface and maxspeed Values

surface_map = {
    'motorway': 'asphalt',
    'trunk': 'asphalt', 
    'trunk_link': 'asphalt', 
    'primary': 'asphalt',
    'secondary': 'paved',
    'tertiary': 'paved',
    'unclassified': 'paved',
    'residential': 'paved',
    'service': 'gravel',
    'track': 'dirt',
    'path': 'dirt',
    'living_street': 'paved'
}

speed_map = {
    'motorway': 110,
    'trunk': 90, 
    'trunk_link': 90, 
    'primary': 90, 
    'secondary': 70,
    'tertiary': 50,
    'unclassified': 50,
    'residential': 30,
    'service': 30,
    'track': 20,
    'path': 10,
    'living_street': 20
}

In [9]:
# Impute missing 'surface' values
gdf['surface_type'] = gdf['surface'].fillna(
    gdf['road_class'].map(surface_map)
)

# Convert 'maxspeed' to numeric (errors='coerce' turns non-numeric values to NaN)
gdf['base_maxspeed_kmh'] = pd.to_numeric(gdf['maxspeed'], errors='coerce')

# 3. Impute missing 'maxspeed' values with the speed map
gdf['base_maxspeed_kmh'] = gdf['base_maxspeed_kmh'].fillna(
    gdf['road_class'].map(speed_map)
)

# Clean up temporary columns
gdf.drop(columns=['surface', 'maxspeed'], inplace=True)
gdf.dropna(subset=['base_maxspeed_kmh', 'surface_type'], inplace=True)

In [10]:
gdf.isna().sum()

osm_id               0
road_class           0
geometry             0
length_km            0
surface_type         0
base_maxspeed_kmh    0
dtype: int64

In [11]:
# Calculate Base Travel Time

gdf['base_maxspeed_kmh'] = gdf['base_maxspeed_kmh'].replace(0, 10) # Set min speed to 10 km/h

gdf['base_travel_time_min'] = (gdf['length_km'] / gdf['base_maxspeed_kmh']) * 60

gdf.columns.tolist()

['osm_id',
 'road_class',
 'geometry',
 'length_km',
 'surface_type',
 'base_maxspeed_kmh',
 'base_travel_time_min']

In [12]:
gdf['is_blocked'] = False

gdf['final_cost'] = gdf['base_travel_time_min'] # Initial cost is just travel time

# Rename geometry column back to 'geometry' and reset CRS to 4326 for GeoJSON export/web map compatibility
gdf = gdf.to_crs('EPSG:4326')

# Select final columns for database ingestion
final_columns = ['osm_id', 'road_class', 'surface_type', 'length_km', 
                 'base_maxspeed_kmh', 'base_travel_time_min', 
                 'is_blocked', 'final_cost', 'geometry']

gdf_final = gdf[final_columns]

output_path = '../datasets/processed/taita_taveta_road_network.geojson'
gdf_final.to_file(output_path, driver='GeoJSON')
print(f"\nSuccessfully processed and saved to: {output_path}")


Successfully processed and saved to: ../datasets/processed/taita_taveta_road_network.geojson


### Processing Flow Accumulation Data

In [13]:
import numpy as np
from osgeo import gdal

In [14]:
fa_file = "../datasets/raw/flow_accumulation_map.tif"
ds = gdal.Open(fa_file)
fa_array = ds.ReadAsArray()

/opt/anaconda3/envs/conda_env/lib/python3.12/site-packages/osgeo/gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [15]:
valid_fa_values = fa_array[fa_array >= 1]

# Calculate the 99th percentile threshold
threshold_percentile = 99
threshold_value = np.percentile(valid_fa_values, threshold_percentile)

print(f"Total valid cells analyzed: {len(valid_fa_values)}")
print(f"The {threshold_percentile}th percentile Flow Accumulation value is: {threshold_value:.2f}")

Total valid cells analyzed: 20123
The 99th percentile Flow Accumulation value is: 912.23


In [16]:
FINAL_THRESHOLD = int(np.ceil(threshold_value)) 
print(f"Using FINAL_THRESHOLD: {FINAL_THRESHOLD}")

Using FINAL_THRESHOLD: 913


In [17]:
import numpy as np
from osgeo import gdal, ogr, osr
import os
import geopandas as gpd

In [18]:
fa_input = "../datasets/raw/flow_accumulation_map.tif"

# Output Temporary Binary Raster (Thresholded)
fa_binary_temp = "../datasets/raw/temp_fa_binary.tif"

constraint_output_geojson = '../datasets/processed/taita_taveta_drainage_constraints.geojson'
constraint_output_shp = '../datasets/processed/taita_taveta_drainage_constraints.shp'

In [19]:
# Open the Flow Accumulation raster
ds = gdal.Open(fa_input)
band = ds.GetRasterBand(1)
fa_array = band.ReadAsArray()

# Filter out NoData values (assuming 1 is the min meaningful value)
valid_fa_values = fa_array[fa_array >= 1]

In [20]:
# Calculate the 99th percentile threshold
# This isolates the top 1% of the drainage network, capturing the main channels.
threshold_percentile = 99
threshold_value = np.percentile(valid_fa_values, threshold_percentile)

In [21]:
# Round up for a clean integer threshold
FINAL_THRESHOLD = int(np.ceil(threshold_value)) 

print(f"Flow Accumulation Min: {fa_array.min()}, Max: {fa_array.max()}")
print(f"Calculated 99th Percentile Threshold: {FINAL_THRESHOLD}")

Flow Accumulation Min: -3.4028234663852886e+38, Max: 4330.0
Calculated 99th Percentile Threshold: 913


### Reclassify (Raster Calculator) to Binary Raster
WE need to create a temporary raster where pixels meeting the threshold are `1` and all others are `0`. This is a crucial intermediate step for `gdal.Polygonize`.

In [22]:
# Create a binary array (True/False based on threshold)
binary_array = (fa_array >= FINAL_THRESHOLD).astype(np.int8)

In [23]:
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create(
    fa_binary_temp, 
    ds.RasterXSize, 
    ds.RasterYSize, 
    1, 
    gdal.GDT_Byte # Use GDT_Byte for a 0/1 binary raster
)
out_ds.SetGeoTransform(ds.GetGeoTransform())
out_ds.SetProjection(ds.GetProjection())

# Write the binary array to the new raster band
out_band = out_ds.GetRasterBand(1)
out_band.WriteArray(binary_array)

0

In [24]:
# Clear variables to flush the output to disk
out_band = None
out_ds = None
ds = None
print(f"Created temporary binary raster: {fa_binary_temp}")

Created temporary binary raster: ../datasets/raw/temp_fa_binary.tif


#### Polygonize the Binary Raster (Raster to Vector)
Now let us use the powerful gdal.Polygonize function to convert the connected areas of 1s into vector polygons.

In [25]:
# Open the binary raster created in Step 3
src_ds = gdal.Open(fa_binary_temp)
src_band = src_ds.GetRasterBand(1)

drv = ogr.GetDriverByName("ESRI Shapefile")

if os.path.exists(constraint_output_shp):
    drv.DeleteDataSource(constraint_output_shp)

dst_ds = drv.CreateDataSource(constraint_output_shp)
dst_layer = dst_ds.CreateLayer('flood_polygons', srs=src_ds.GetSpatialRef())

In [26]:
# Add a field to store the raster value (which will be '1' for all polygons)
# This is required by the Polygonize function
fd_defn = ogr.FieldDefn('R_VALUE', ogr.OFTInteger)
dst_layer.CreateField(fd_defn)
dst_field_index = dst_layer.GetLayerDefn().GetFieldIndex('R_VALUE')

In [27]:
# --- Run the Polygonization ---
# gdal.Polygonize( src_band, mask_band, output_layer, pixel_value_field_index, options, callback )
gdal.Polygonize(
    src_band, 
    None,  # No mask band used
    dst_layer, 
    dst_field_index, 
    []
)

0

In [28]:
# Close the datasets to ensure everything is written to disk
dst_ds = None
src_ds = None
print(f"Successfully polygonized into: {constraint_output_shp}")

Successfully polygonized into: ../datasets/processed/taita_taveta_drainage_constraints.shp


#### Final Cleanup and Schema Assignment (GeoPandas)
Load the resulting Shapefile back into GeoPandas for schema cleanup and export to GeoJSON

In [29]:
gdf_constraints = gpd.read_file(constraint_output_shp)
gdf_constraints.tail()

,R_VALUE,geometry
78,1,"POLYGON ((38.78333 -3.95, 38.79167 -3.95, 38.7..."
79,1,"POLYGON ((38.79167 -3.96667, 38.8 -3.96667, 38..."
80,1,"POLYGON ((38.8 -3.975, 38.80833 -3.975, 38.808..."
81,1,"POLYGON ((38.80833 -3.99167, 38.84167 -3.99167..."
82,0,"POLYGON ((37.59167 -2.68333, 39.21667 -2.68333..."


In [30]:
# Assign the required Mwamko AI schema attributes
gdf_constraints = gdf_constraints[gdf_constraints['R_VALUE'] == 1].copy()

if len(gdf_constraints) == 0:
    print("Warning: Filtering removed all polygons. Check your threshold value!")
else:
    print(f"Removed bounding box polygon. Remaining flood features: {len(gdf_constraints)}")

    
gdf_constraints['constraint_type'] = 'STATIC_DRAINAGE_NETWORK' 
gdf_constraints['priority_level'] = 10 # Default priority level for static constraints
gdf_constraints['county_id'] = 'Taita_Taveta'
gdf_constraints['active_until'] = pd.NaT # Not applicable for permanent feature

Removed bounding box polygon. Remaining flood features: 82


In [31]:
# Drop the temporary Raster Value column
gdf_constraints.drop(columns=['R_VALUE'], inplace=True)

In [32]:
# Select final columns and ensure CRS is WGS 84 for web compatibility
gdf_constraints = gdf_constraints[['constraint_type', 'priority_level', 'county_id', 'active_until', 'geometry']]
gdf_constraints = gdf_constraints.to_crs('EPSG:4326')

In [33]:
gdf_constraints.to_file(constraint_output_geojson, driver='GeoJSON')
print(f"Final constraint data saved to: {constraint_output_geojson}")

Final constraint data saved to: ../datasets/processed/taita_taveta_drainage_constraints.geojson


In [34]:
gdf_constraints.tail()

,constraint_type,priority_level,county_id,active_until,geometry
77,STATIC_DRAINAGE_NETWORK,10,Taita_Taveta,NaT,"POLYGON ((38.59167 -3.94167, 38.6 -3.94167, 38..."
78,STATIC_DRAINAGE_NETWORK,10,Taita_Taveta,NaT,"POLYGON ((38.78333 -3.95, 38.79167 -3.95, 38.7..."
79,STATIC_DRAINAGE_NETWORK,10,Taita_Taveta,NaT,"POLYGON ((38.79167 -3.96667, 38.8 -3.96667, 38..."
80,STATIC_DRAINAGE_NETWORK,10,Taita_Taveta,NaT,"POLYGON ((38.8 -3.975, 38.80833 -3.975, 38.808..."
81,STATIC_DRAINAGE_NETWORK,10,Taita_Taveta,NaT,"POLYGON ((38.80833 -3.99167, 38.84167 -3.99167..."


In [35]:
gdf_constraints.explore()

### **Upload Datasets to Mwamko Database**

In [36]:
import geopandas as gpd
from sqlalchemy import create_engine, text
from sqlalchemy.types import BigInteger
import pandas as pd

In [37]:
# Define the connection parameters for the mwamko user
DB_USER = "mwamko"
DB_PASSWORD = "mwamkoai2025"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "mwamko_db"

# Create the SQLAlchemy connection string
conn_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_string)

print("Database connection engine created successfully.")

Database connection engine created successfully.


In [38]:
ROAD_FILE = "../datasets/processed/taita_taveta_road_network.geojson"
CONSTRAINT_FILE = "../datasets/processed/taita_taveta_drainage_constraints.geojson"

In [39]:
# Read the road data
gdf_roads = gpd.read_file(ROAD_FILE)

gdf_roads = gdf_roads.to_crs(epsg=4326)

# Use osm_id as the primary key. We must rename it to 'id' if using the standard GeoPandas method.
gdf_roads.rename(columns={'osm_id': 'id'}, inplace=True)

In [40]:
try:
    gdf_roads.to_postgis(
        name='road_segments', 
        con=engine, 
        if_exists='replace',
        index=False,
        chunksize=1000,
        dtype={
            'id': BigInteger, # Use SQLAlchemy's BigInteger class
            'geometry': 'GEOMETRY(LineString, 4326)'
        }
    )
    
    with engine.connect() as connection:
        connection.execute(
            text('ALTER TABLE road_segments ADD PRIMARY KEY (id);')
        )
        connection.commit()
        
    print(f"Successfully uploaded {len(gdf_roads)} road segments to 'road_segments'.")
    
except Exception as e:
    print(f"Error uploading road segments: {e}")

Successfully uploaded 9895 road segments to 'road_segments'.


In [41]:
gdf_constraints = gpd.read_file(CONSTRAINT_FILE)

# Ensure data is in the required EPSG:4326 CRS
gdf_constraints = gdf_constraints.to_crs(epsg=4326)

In [42]:
try:
    gdf_constraints.to_postgis(
        name='constraints', 
        con=engine, 
        if_exists='replace', 
        index=False,
        chunksize=500,
        dtype={'geometry': 'GEOMETRY(Polygon, 4326)'}
    )
    
    print(f"Successfully uploaded {len(gdf_constraints)} constraint polygons to 'constraints'.")
    
except Exception as e:
    print(f"Error uploading constraints: {e}")

Successfully uploaded 82 constraint polygons to 'constraints'.


In [43]:
# Verify
from sqlalchemy import text

try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT count(*) FROM constraints;"))
        count = result.scalar()
        print(f"Database verification: 'constraints' table has {count} rows.")
        
except Exception as e:
    print(f"Verification Error: {e}")

Database verification: 'constraints' table has 82 rows.


In [44]:
from sqlalchemy import text

try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT count(*) FROM road_segments;"))
        count = result.scalar()
        print(f"Database verification: 'road_segments' table has {count} rows.")
        
except Exception as e:
    print(f"Verification Error: {e}")

Database verification: 'road_segments' table has 9895 rows.
